## **Greedy Autoregressive Text Generation**


In [6]:
import numpy as np

def generate_greedy(model, idx: list, max_new_tokens: int, context_size: int) -> list:

    idx = np.array([idx])

    for _ in range(max_new_tokens):

        # Crop to the last context_size tokens
        idx_cond = idx[:, -context_size:]

        # Get logits from the model
        logits = model(idx_cond)

        # Take logits from the last time step
        logits = logits[:, -1, :]

        # Pick the token with the highest logit
        next_token = np.argmax(logits, axis=-1)

        # Append the new token
        idx = np.concatenate([idx, next_token[:, None]], axis=1)

    return idx[0].tolist()


def model(x):
    V = 5
    T = x.shape[1]
    logits = np.zeros((1, T, V))
    for t in range(T):
        nxt = (int(x[0, t]) + 1) % V
        logits[0, t, nxt] = 1.0
    return logits
print(generate_greedy(model, [0], 4, 8))

[0, 1, 2, 3, 4]


## **Byte Pair Encoding (BPE) Tokenizer**

## **What does BPE do?**

**Byte Pair Encoding (BPE)** is a subword tokenization algorithm.

The main idea is simple:

> **BPE looks for the most frequent adjacent pair of tokens and merges them into a single token.**

It starts with small tokens, such as characters, and gradually combines them into larger **subword tokens**.

```text
Small tokens
		↓
Frequent pairs
		↓
Merge the pairs
		↓
Larger subwords
```

---

## **Example**

Consider the following corpus:

```python
corpus = {
		"h u g </w>": 10,
		"p u g </w>": 5,
		"p u g s </w>": 5
}
```

The numbers represent how many times each word appears in the corpus.

So:

```text
h u g </w>       ×10
p u g </w>       ×5
p u g s </w>     ×5
```

`</w>` represents the **end of the word**.

---

## **Step 1 — Count Adjacent Pairs**

BPE looks at every pair of neighboring tokens.

For example:

```text
h u g </w>
↑ ↑
(h, u)

h u g </w>
	↑ ↑
(u, g)

h u g </w>
		↑ ↑
(g, </w>)
```

We do this for every word and multiply each occurrence by the word frequency.

### **Example: `(u, g)`**

The pair `(u, g)` appears in all three words:

```text
h u g </w>       ×10
	↑ ↑

p u g </w>       ×5
	↑ ↑

p u g s </w>     ×5
	↑ ↑
```

Therefore:

```text
(u, g) = 10 + 5 + 5 = 20
```

Some other pair counts are:

```text
(h, u)      = 10
(p, u)      = 5 + 5 = 10
(g, </w>)   = 10 + 5 = 15
(g, s)      = 5
(s, </w>)   = 5
```

So we have:

```text
(u, g)      → 20  ← most frequent
(g, </w>)   → 15
(h, u)      → 10
(p, u)      → 10
(g, s)      → 5
(s, </w>)   → 5
```

---

## **Step 2 — Find the Most Frequent Pair**

The most frequent pair is:

```text
(u, g)
```

because it has the highest frequency:

```text
(u, g) → 20
```

BPE therefore decides to merge:

```text
u + g → ug
```

---

## **Step 3 — Merge the Pair Everywhere**

We replace every occurrence of `(u, g)` with `ug`.

Before:

```text
h u g </w>
p u g </w>
p u g s </w>
```

After:

```text
h ug </w>
p ug </w>
p ug s </w>
```

Notice that **all occurrences** of `(u, g)` are merged.

---

## **Step 4 — Repeat**

BPE now counts the adjacent pairs again.

The corpus is now:

```text
h ug </w>       ×10
p ug </w>       ×5
p ug s </w>     ×5
```

Now `(ug, </w>)` appears in the first two words:

```text
h ug </w>       ×10
	↑  ↑

p ug </w>       ×5
	↑  ↑
```

Therefore:

```text
(ug, </w>) = 10 + 5 = 15
```

This is now the most frequent pair.

So BPE merges:

```text
ug + </w> → ug</w>
```

---

## **Final Result**

If we perform `2` merge operations, the learned merges are:

```python
[
		("u", "g"),
		("ug", "</w>")
]
```

The important thing is that BPE is **learning which tokens should be combined based on their frequency in the corpus**.

---

## **The Big Picture**

BPE repeatedly performs these three steps:

```text
1. Count adjacent pairs
				↓
2. Find the most frequent pair
				↓
3. Merge that pair
				↓
			Repeat
```

Starting from:

```text
h u g </w>
```

we can get:

```text
u + g
	↓
ug
```

and then:

```text
ug + </w>
		↓
ug</w>
```

So BPE gradually builds larger and more useful **subword tokens** from smaller tokens.

---

## **Why is this useful?**

Instead of treating every complete word as a completely separate token, BPE can learn common parts of words.

For example:

```text
playing
played
player
plays
```

can share the common subword:

```text
play
```

and then have different endings:

```text
play + ing
play + ed
play + er
play + s
```

This gives BPE a useful balance between:

* **Character-level tokenization** → small vocabulary but very long sequences
* **Word-level tokenization** → large vocabulary and problems with unknown words
* **Subword tokenization** → reusable pieces of words

### **In one sentence:**

> **BPE starts with small tokens and repeatedly merges the most frequent adjacent pair to build larger subword tokens.**


In [4]:
def byte_pair_encoding(corpus: dict, num_merges: int) -> list:
    """
    Train a BPE tokenizer on the given corpus.
    """

    # Store the merges
    merges = []

    # Repeat the process for the requested number of merges
    for _ in range(num_merges):

        # Count all adjacent pairs
        pair_counts = {}

        for word, frequency in corpus.items():
            tokens = word.split()

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                pair_counts[pair] = pair_counts.get(pair, 0) + frequency

        # Stop if there are no more pairs to merge
        if not pair_counts:
            break

        # Find the most frequent pair
        best_pair = max(pair_counts, key=pair_counts.get)

        # Save the merge
        merges.append(best_pair)

        # Merge the best pair in every word
        new_corpus = {}

        for word, frequency in corpus.items():
            tokens = word.split()

            new_tokens = []
            i = 0

            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and (tokens[i], tokens[i + 1]) == best_pair
                ):
                    new_tokens.append(tokens[i] + tokens[i + 1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            # Convert tokens back to a string
            new_word = " ".join(new_tokens)

            # Keep the original frequency
            new_corpus[new_word] = frequency

        # Update the corpus for the next merge
        corpus = new_corpus

    return merges


result = byte_pair_encoding({"h u g </w>": 10, "p u g </w>": 5, "p u g s </w>": 5}, 2)
print(result)

[('u', 'g'), ('ug', '</w>')]


## **Matrix-Vector Dot Product**

In [7]:
import numpy as np 
def matrix_dot_vector(a: list[list[int|float]], b: list[int|float]) -> list[int|float]:
	# Return a list where each element is the dot product of a row of 'a' with 'b'.
	# If the number of columns in 'a' does not match the length of 'b', return -1.
	a = np.array(a)
	b = np.array(b)
	if a.shape[1] != b.shape[0]:
		return -1
	return (a @ b).tolist()	

print(matrix_dot_vector([[1, 2, 3], [2, 4, 5], [6, 8, 9]], [1, 2, 3]))

[14, 25, 49]


## **Linear Regression Using Normal Equation**

In [8]:
import numpy as np
def linear_regression_normal_equation(X: list[list[float]], y: list[float]) -> list[float]:
    X , y = np.array(X) , np.array(y)

    theta = np.linalg.inv(X.T @ X) @ X.T @ y

    return np.round(theta, 4).tolist()

print(linear_regression_normal_equation([[1, 1], [1, 2], [1, 3]], [1, 2, 3]))

[-0.0, 1.0]


## **Embedding Layer as One-Hot Matrix Multiplication**

In [10]:
import numpy as np

def embedding_via_one_hot(token_ids, W):
    """
    Compute token embeddings via one-hot encoding and matrix multiplication.

    Args:
        token_ids: list or 1D array of integer token IDs
        W: numpy array of shape (vocab_size, embed_dim)

    Returns:
        numpy array of shape (len(token_ids), embed_dim)
    """
    num_tokens = len(token_ids)     # 3
    vocab_size = W.shape[0]  # 4
    H = np.zeros((num_tokens, vocab_size))

    for i in range(num_tokens):
        H[i, token_ids[i]] = 1

    return H @ W


W = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0],
              [7.0, 8.0, 9.0],
              [10.0, 11.0, 12.0]])
print(embedding_via_one_hot([2, 0, 3], W).tolist())

[[7.0, 8.0, 9.0], [1.0, 2.0, 3.0], [10.0, 11.0, 12.0]]


## **Calculate Cosine Similarity Between Vectors**

In [11]:
import numpy as np

def cosine_similarity(v1, v2):
    v1 = np.array(v1)
    v2 = np.array(v2)
    
    if len(v1) == 0 or len(v2) == 0:
        raise ValueError("Vectors cannot be empty")
    if v1.shape != v2.shape:
        raise ValueError("Vectors must have the same shape")
    
    dot_product = np.dot(v1, v2)
    magnitude = np.linalg.norm(v1) * np.linalg.norm(v2)
    
    return dot_product / magnitude


v1 = np.array([1, 2, 3])
v2 = np.array([2, 4, 6])
print(round(cosine_similarity(v1, v2), 3))


1.0


## **Dot Product Calculator**

In [13]:
import numpy as np

def calculate_dot_product(vec1, vec2):
	"""
	Calculate the dot product of two vectors.
	Args:
		vec1 (numpy.ndarray): 1D array representing the first vector.
		vec2 (numpy.ndarray): 1D array representing the second vector.
	Returns:
		The dot product of the two vectors.
	"""
	# Your code here
	vec1 = np.array(vec1)
	vec2 = np.array(vec2)
	return (vec1 @ vec2).tolist()

print(calculate_dot_product(np.array([1, 2, 3]), np.array([4, 5, 6])))

32


## **Calculate Number of Parameters in Neural Network**

In [15]:
def calculate_parameters(layers: list[dict]) -> int:
	"""
	Calculate the total number of trainable parameters in a neural network.

	Args:
		layers: List of dictionaries, each describing a layer.

	Returns:
		Total number of trainable parameters (int).
	"""
	# Your code here
	total = 0

	for layer in layers : 

		if layer['type'] == 'dense':
			w = layer['input_size'] * layer['output_size']
			if layer.get('bias', True):
				w += layer['output_size']
			total += w

		elif layer['type'] == 'conv2d':
			w = layer['in_channels'] * layer['out_channels'] * layer['kernel_size']**2
			if layer.get('bias', True):
				w += layer['out_channels']
			total += w
	return total

print(calculate_parameters([{'type': 'dense', 'input_size': 784, 'output_size': 128, 'bias': True}, {'type': 'dense', 'input_size': 128, 'output_size': 10, 'bias': True}]))

101770


## **Transpose of a Matrix**

In [16]:
import numpy
def transpose_matrix(a: list[list[int|float]]) -> list[list[int|float]]:
    """
    Transpose a 2D matrix by swapping rows and columns.
    
    Args:
        a: A 2D matrix of shape (m, n)
    
    Returns:
        The transposed matrix of shape (n, m)
    """
    # Your code here
    return [list(row) for row in zip(*a)]

print(transpose_matrix([[1, 2], [3, 4], [5, 6]]))

[[1, 3, 5], [2, 4, 6]]


## **Softmax Activation Function Implementation**

In [17]:
import math
import numpy as np 
def softmax(scores):
	maxx = np.max(scores)     
	denom = np.sum([math.exp(z - maxx) for z in scores])       # compute once, using ALL scores
	
	results = []
	for score in scores:
		result = float(math.exp(score - maxx) / denom )   # now use maxx and denom here
		results.append(result)
	return results

print([round(x, 4) for x in softmax([1, 2, 3])])

[0.09, 0.2447, 0.6652]


## **Temperature Sampling**

In [18]:
import numpy as np

def temperature_sampling(logits: np.ndarray, temperature: float) -> list:
	"""
	Compute temperature-scaled softmax probabilities from logits.
	
	Args:
		logits: 1D numpy array of raw model output scores
		temperature: float controlling distribution sharpness
	
	Returns:
		List of probabilities after temperature scaling
	"""
	
	# Temperature <= 0 → greedy / one-hot distribution
	if temperature <= 0:
		probs = np.zeros_like(logits, dtype=float)
		probs[np.argmax(logits)] = 1.0
		return probs.tolist()

	# Scale logits by temperature
	scaled_logits = logits / temperature

	# Numerical stability: subtract the maximum
	scaled_logits = scaled_logits - np.max(scaled_logits)

	# Exponentiate
	exp_logits = np.exp(scaled_logits)

	# Normalize
	probs = exp_logits / np.sum(exp_logits)

	return probs.tolist()

result = temperature_sampling(np.array([1.0, 2.0, 3.0]), 1.0)
print([round(x, 4) for x in result])

[0.09, 0.2447, 0.6652]
